In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
      .appName("Olist Ecommerce Performance Optimization")\
        .getOrCreate()

In [0]:
# spark = SparkSession.builder\
#     .appName("Olist Ecommerce Performance Optimization")\
#     .config('spark.executor.memory','6g')\
#     .config('spark.executor.instances','2')\
#     .config('spark.driver.memory','4g')\
#     .config('spark.driver.maxResultSize','2g')\
#     .config('spark.sql.shuffle.partitions','64')\
#     .config('spark.default.parallelism','64')\
#     .config('spark.sql.adaptive.enabled','true')\
#     .config('spark.sql.adaptive.coalescePartition.enabled','true')\
#     .config('spark.sql.autoBroadcastJoinThreshold',20*1024*1024)\
#     .config('spark.sql.files.maxPartitionBytes','64MB')\
#     .config('spark.sql.files.openCostInBytes','2MB')\
#     .config('spark.memory.fraction',0.8)\
#     .config('spark.memory.storageFraction',0.2)\
#     .getOrCreate()


    #i guess o need of setting manually in Databricks


In [0]:

#connect ADLSgen2 to Databricks

spark.conf.set("fs.azure.account.key.adlsgen2spark01.dfs.core.windows.net",
                 "Storageaccount -> Security + Networking -> Access Keys-->Key1valuePaste here ")
#basePath
adlsgen2CSVpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/csv/"
adlsgen2PARQUETpath = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/parquet/"

#read data
customers_df = spark.read.csv(adlsgen2CSVpath + "olist_customers_dataset", header=True, inferSchema=True) 
geolocation_df = spark.read.csv(adlsgen2CSVpath + "olist_geolocation_dataset", header=True, inferSchema=True) 
order_items_df = spark.read.csv(adlsgen2CSVpath + "olist_order_items_dataset", header=True, inferSchema=True) 
payments_df = spark.read.csv(adlsgen2CSVpath + "olist_order_payments_dataset", header=True, inferSchema=True) 
reviews_df = spark.read.csv(adlsgen2CSVpath + "olist_order_reviews_dataset", header=True, inferSchema=True) 
orders_df = spark.read.csv(adlsgen2CSVpath + "olist_orders_dataset", header=True, inferSchema=True) 
products_df = spark.read.csv(adlsgen2CSVpath + "olist_products_dataset", header=True, inferSchema=True) 
sellers_df = spark.read.csv(adlsgen2CSVpath + "olist_sellers_dataset", header=True, inferSchema=True) 
catgeory_translation_df = spark.read.csv(adlsgen2CSVpath + "product_category_name_translation", header=True, inferSchema=True) 

In [0]:
adlsgen2_gold_parquet = "abfss://olist-data@adlsgen2spark01.dfs.core.windows.net/gold/"
full_orders_df = spark.read.parquet(adlsgen2_gold_parquet+'full_orders_df.parquet')

#As Reading in Parquet format - no need to worry about schema and all & its optimized also

In [0]:
full_orders_df.show()

+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-----+-------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+--------------------+------------+--------------------+------------------------+-------------------+--------------+---------------------------+-------------------+-------------------+-------------------+-----------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+------------------+------------+--------------------+-------------+------------+-----------+------------------+----------------+-----------+--------------+----------------+
|         custom

#Optimized Join Strategy

In [0]:
#BroadCast
from pyspark.sql.functions import broadcast
 
customers_broadcast_df = broadcast(customers_df)
optimized_broadcast_join_df = full_orders_df.join(customers_broadcast_df,'customer_id')

In [0]:
#Sort and Merge Join

sorted_customer_df = customers_df.sortWithinPartitions('customer_id')
sorted_orders_df = full_orders_df.sortWithinPartitions('customer_id')

optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customer_df,'customer_id')

In [0]:
#Bucket Join

bucketed_customers_df = customers_df.repartition(10,'customer_id')
bucketed_orders_df = full_orders_df.repartition(10,'customer_id')

bucket_join_df = bucketed_orders_df.join(bucketed_customers_df,'customer_id')

In [0]:
#Skew Join Handling

skew_handled_join = full_orders_df.join(customers_df.hint('skew'),'customer_id')